# Perpendicular Shift Growth Along a 2D Flow Trajectory

This notebook demonstrates how the perpendicular displacement of a nearby trajectory grows
along a field line / 2D flow trajectory. It is a simplified (synthetic) example — not using real W7-X data.

**Core concept**: Given a 2D flow $\dot{X} = B(X)$, compute how an infinitesimal perpendicular offset
to the flow grows. This is related to the finite-time Poincaré transversality (FPT) concept in `pyna`.

**Source**: Adapted from `MCF_scripts/how_the_perpendicular_shift_grows_along_a_trajectory_of_2D_flows.ipynb`

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

## Define a synthetic 2D flow

We use a simple non-trivial flow:
$$\dot{x} = \frac{\sin x}{2} + \frac{\sin y}{2} + 1.5, \quad \dot{y} = 0.2 y$$

In [ ]:
def f(t, Xpol):
    x, y = Xpol
    dXpoldt = [np.sin(x)/2 + np.sin(y)/2 + 1.5, 0.2*y]
    return dXpoldt

Xpol0 = [0.0, -1.0]
t_span = [0.0, 32]
Xpol_sol = solve_ivp(f, t_span, Xpol0, rtol=1e-10, atol=1e-13, dense_output=True)
print(f"Integration successful: {Xpol_sol.success}")

## Compute the Jacobian (tangent map)

The Jacobian $DX$ satisfies:
$$\dot{DX} = \frac{\partial B}{\partial X}(X(t)) \cdot DX(t), \quad DX(0) = I$$

In [ ]:
def dDXpoldt(t, DXpol):
    x, y = Xpol_sol.sol(t)
    pBpolpRpZ = np.asarray([
        [np.cos(x)/2, np.cos(y)/2],
        [0.0,         0.2        ]
    ])
    dDXpoldt = pBpolpRpZ @ np.reshape(DXpol, (2, 2))
    return np.reshape(dDXpoldt, (4,))

DXpol_sol = solve_ivp(dDXpoldt, t_span, [1.0, 0.0, 0.0, 1.0], rtol=1e-10, atol=1e-13)
print(f"Jacobian integration successful: {DXpol_sol.success}")

## Compute perpendicular shift growth (scalar)

The perpendicular shift $x_\perp$ satisfies:
$$\dot{x}_\perp = \left(\frac{\partial B}{\partial X} \hat{b}_\perp\right) \cdot \hat{b}_\perp \cdot x_\perp$$

where $\hat{b}_\perp = R_{90°} \hat{B} / |\hat{B}|$ is the unit vector perpendicular to the flow.

In [ ]:
def dxperpdt(t, xperp):
    x, y = Xpol_sol.sol(t)
    Bpol = np.asarray([np.sin(x)/2 + np.sin(y)/2 + 1.5, 0.2*y])
    Bperp = np.array([[0, -1], [1, 0]]) @ Bpol
    Bperp_unit = Bperp / np.linalg.norm(Bperp)

    pBpolpRpZ = np.asarray([
        [np.cos(x)/2, np.cos(y)/2],
        [0.0,         0.2        ]
    ])
    Bpol_change_along_Bperp = pBpolpRpZ @ Bperp_unit
    Bpol_perp_component_change_along_Bperp = Bpol_change_along_Bperp @ Bperp_unit
    return Bpol_perp_component_change_along_Bperp * xperp

xperp_sol = solve_ivp(dxperpdt, t_span, [1.0], rtol=1e-10, atol=1e-13)
print(f"Perpendicular shift at t=T: {xperp_sol.y[0, -1]:.6f}")

## Visualize the trajectory and neighbor convergence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: flow trajectory
ax = axes[0]
ax.plot(Xpol_sol.y[0, :], Xpol_sol.y[1, :], 'b-', label='Trajectory')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('2D Flow Trajectory')
ax.legend()
ax.grid(True)

# Right: neighbor trajectories (shifted perpendicularly)
ax = axes[1]
Xpol_traj_num = 10
Xpol0_shift = np.asarray([0.1, -0.09]) * 1e-6
endpoints = np.zeros((2, Xpol_traj_num))
for i in range(Xpol_traj_num):
    sol_temp = solve_ivp(f, t_span, Xpol0 + Xpol0_shift * i / Xpol_traj_num, rtol=1e-10, atol=1e-13)
    endpoints[:, i] = sol_temp.y[:, -1]

ax.scatter(endpoints[0, :], endpoints[1, :], c=range(Xpol_traj_num), cmap='viridis', label='Endpoints')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Neighbor Trajectory Endpoints at t=T')
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.show()

print(f"Perpendicular growth factor (scalar ODE): {xperp_sol.y[0, -1]:.6f}")
print(f"Perpendicular growth factor (full Jacobian): {np.linalg.norm(np.reshape(DXpol_sol.y[:, -1], (2,2)) @ Xpol0_shift):.6f}")